In [ ]:
import torch
from torch import nn, Tensor
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from IPython.display import display, clear_output

In [ ]:
# Download MNIST data
batch_size = 128    # 256

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
)

# Train a label-conditional CNN flow model for MNIST

In [ ]:
class Flow(nn.Module):
    """Compact label-conditional U-Net-style CNN vector field."""

    def __init__(
        self,
        num_classes: int = 10,
        hidden_size: int = 16,
    ):
        super().__init__()

        if hidden_size % 4 != 0:
            raise ValueError("hidden_size must be divisible by 4")

        self.num_classes = num_classes
        condition_channels = 1 + 1 + num_classes
        h = hidden_size

        self.encoder_14 = nn.Sequential(
            nn.Conv2d(condition_channels, h, kernel_size=5, stride=2, padding=2),
            nn.GroupNorm(4, h),
            nn.SiLU(),
        )
        self.encoder_7 = nn.Sequential(
            nn.Conv2d(h, 2 * h, kernel_size=5, stride=2, padding=2),
            nn.GroupNorm(4, 2 * h),
            nn.SiLU(),
        )
        self.bottleneck = nn.Sequential(
            nn.Conv2d(2 * h, 4 * h, kernel_size=3, padding=1),
            nn.GroupNorm(4, 4 * h),
            nn.SiLU(),
        )
        self.up_14 = nn.ConvTranspose2d(
            4 * h, 2 * h, kernel_size=4, stride=2, padding=1
        )
        self.decoder_14 = nn.Sequential(
            nn.Conv2d(3 * h, 2 * h, kernel_size=3, padding=1),
            nn.GroupNorm(4, 2 * h),
            nn.SiLU(),
        )
        self.up_28 = nn.ConvTranspose2d(
            2 * h, h, kernel_size=4, stride=2, padding=1
        )
        self.decoder_28 = nn.Sequential(
            nn.Conv2d(h + condition_channels, h, kernel_size=3, padding=1),
            nn.GroupNorm(4, h),
            nn.SiLU(),
            nn.Conv2d(h, 1, kernel_size=3, padding=1),
        )

    def forward(
        self,
        t: Tensor,
        c: Tensor,
        x_t: Tensor,
    ) -> Tensor:
        if x_t.ndim != 4 or x_t.shape[1:] != (1, 28, 28):
            raise ValueError("x_t must have shape (batch, 1, 28, 28)")

        t = t.reshape(-1, 1).to(device=x_t.device, dtype=x_t.dtype)
        c = c.reshape(-1).to(device=x_t.device)
        if t.shape != (len(x_t), 1) or c.shape != (len(x_t),):
            raise ValueError("t or c has an incompatible batch dimension")

        class_code = F.one_hot(c.long(), num_classes=self.num_classes).to(x_t.dtype)
        class_maps = class_code[:, :, None, None].expand(-1, -1, 28, 28)
        time_map = t[:, :, None, None].expand(-1, -1, 28, 28)
        conditioned = torch.cat((x_t, time_map, class_maps), dim=1)

        encoded_14 = self.encoder_14(conditioned)
        encoded_7 = self.encoder_7(encoded_14)
        bottleneck = self.bottleneck(encoded_7)
        decoded_14 = self.decoder_14(
            torch.cat((self.up_14(bottleneck), encoded_14), dim=1)
        )
        decoded_28 = self.up_28(decoded_14)
        return self.decoder_28(torch.cat((decoded_28, conditioned), dim=1))

    def step(
        self,
        x_t: Tensor,
        t_start: Tensor,
        t_end: Tensor,
        c: Tensor,
    ) -> Tensor:
        batch_t_start = t_start.reshape(1, 1).expand(x_t.shape[0], 1)
        dt = t_end - t_start

        # RK2 midpoint method
        k1 = self(
            t=batch_t_start,
            c=c,
            x_t=x_t,
        )

        x_mid = x_t + dt * k1 / 2
        t_mid = batch_t_start + dt / 2

        k2 = self(
            t=t_mid,
            c=c,
            x_t=x_mid,
        )

        return x_t + dt * k2

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

In [ ]:
flow = Flow().to(device)

optimizer = torch.optim.Adam(flow.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

loss_history = []

fig, ax = plt.subplots(figsize=(7, 4))

n_epochs = 10
plot_every = 50
step = 0

for epoch in range(n_epochs):

    for images, c in loader:

        # MNIST image: (B, 1, 28, 28)
        images = images.to(device)
        c = c.to(device)

        # Keep the image grid for the CNN and map [0,1] -> [-1,1]
        x_1 = 2 * images - 1

        # Gaussian source
        x_0 = torch.randn_like(x_1)

        # One random time for each (x_0, x_1) pair
        t = torch.rand(len(x_1), 1, device=device)
        t_image = t[:, :, None, None]

        # Linear interpolation path
        x_t = (1 - t_image) * x_0 + t_image * x_1

        # Target velocity
        dx_t = x_1 - x_0

        optimizer.zero_grad()

        loss = loss_fn(
            flow(
                t=t,
                c=c,
                x_t=x_t,
            ),
            dx_t,
        )

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        step += 1

        if step % plot_every == 0:
            ax.clear()
            ax.plot(loss_history)
            ax.set_xlabel("Iteration")
            ax.set_ylabel("Flow-matching loss")
            ax.set_title(
                f"Epoch {epoch + 1}, iteration {step}, "
                f"loss = {loss.item():.4f}"
            )

            clear_output(wait=True)
            display(fig)

plt.close(fig)

# Sample new (fake) MNIST-style digits

In [ ]:
labels = torch.tensor(
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    device=device,
)

# Start with one Gaussian noise image for each requested digit
n_samples = len(labels)

x = torch.randn(
    n_samples,
    1,
    28,
    28,
    device=device,
)

# Number of displayed states, including the initial noise and final sample.
# The trajectory tensor below therefore has shape (n_steps, 10, 1, 28, 28).
n_steps = 10

time_steps = torch.linspace(
    0,
    1,
    n_steps,
    device=device,
)

flow.eval()

trajectory = [x.detach().cpu()]

with torch.no_grad():

    for i in range(n_steps - 1):

        x = flow.step(
            x_t=x,
            t_start=time_steps[i],
            t_end=time_steps[i + 1],
            c=labels,
        )
        trajectory.append(x.detach().cpu())

# Stack the full path and map [-1,1] back to [0,1] for display.
trajectory = torch.stack(trajectory)
trajectory_display = ((trajectory + 1) / 2).clamp(0, 1).squeeze(2)

# The last row contains the final generated digits.
generated = trajectory_display[-1]

# Display an n_steps x 10 matrix: rows are times and columns are labels.
fig, axes = plt.subplots(
    n_steps,
    10,
    figsize=(15, 1.35 * n_steps),
    squeeze=False,
)

time_steps_cpu = time_steps.detach().cpu()
labels_cpu = labels.detach().cpu()

for row in range(n_steps):
    for col in range(10):
        ax = axes[row, col]
        ax.imshow(
            trajectory_display[row, col],
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        ax.set_xticks([])
        ax.set_yticks([])

        if row == 0:
            ax.set_title(f"digit {labels_cpu[col].item()}")
        if col == 0:
            ax.set_ylabel(
                f"t={time_steps_cpu[row].item():.2f}",
                rotation=0,
                labelpad=25,
                va="center",
            )

fig.suptitle("Conditional flow-matching trajectories: noise to digits", y=1.002)
plt.tight_layout()
plt.show()